# **Requirements Gathering**

In [ ]:


!pip install transformers accelerate torch datasets pandas matplotlib ipywidgets --quiet

from huggingface_hub import login
import torch, gc, pandas as pd, time, matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


login()

!nvidia-smi

def clear_memory():
    """Safely clears GPU cache and garbage."""
    gc.collect()
    torch.cuda.empty_cache()
    print("✅ GPU memory cleared.")

def generate_code(model_id, prompt, max_new_tokens=256):
    """
    Generate code using a given Hugging Face model (optimized for Colab T4 GPU).
    - Loads model & tokenizer
    - Uses fp16 + device_map='auto' for efficiency
    - Generates code for a single prompt
    """
    print(f"\n🚀 Loading model: {model_id}")
    start_time = time.time()

    # Load model + tokenizer (fp16 precision for reduced memory)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    # Prepare pipeline
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    # Generate code
    print(f"⚙️ Generating code for prompt: {prompt[:60]}...")
    output = pipe(prompt, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.3)[0]['generated_text']

    # Track metrics
    end_time = time.time()
    duration = round(end_time - start_time, 2)

    print(f"Done in {duration}s")
    clear_memory()

    return {"model": model_id, "prompt": prompt, "output": output, "time": duration}


#**10 Promts Feeding**

In [ ]:


prompts = [
    "Write a program that checks whether a string or number entered by the user is a palindrome.",
    "Write a program to check if a number is an Armstrong number. (An n-digit number is Armstrong if the sum of its digits raised to n equals the number itself.)",
    "Write a program that prints all prime numbers between two numbers provided by the user.",
    "Write a program that calculates the sum of digits of a number entered by the user.",
    "Write a program that prints a number pyramid for n rows:\n1\n12\n123\n1234\n...",
    "Write a program that sorts a list of numbers entered by the user without using the built-in sort() function.",
    "Write a program that counts the frequency of each word in a sentence entered by the user.",
   "Create a program that simulates an ATM with these features: check balance, deposit, withdraw, and exit.",
   "Write a program that prints the first n Fibonacci numbers using recursion.",
  "Write a program that adds two matrices entered by the user."

]

for i, p in enumerate(prompts, 1):
    print(f"{i}. {p}")


# **Individual Model Execution and Evaluation**

# **Model 1 : Deep seek**

In [ ]:

import time, torch, gc, pandas as pd, matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()

model_id = "deepseek-ai/deepseek-coder-1.3b-instruct"
print(f"Loading {model_id} ...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, torch_dtype=torch.float16, device_map="auto")

results = []
for i, prompt in enumerate(prompts, 1):
    print(f"\n==============================")
    print(f"🧠 Prompt {i}: {prompt}")
    print(f"==============================")

    start = time.time()
    output = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.3)[0]['generated_text']
    end = time.time()

    duration = round(end - start, 2)
    print(f"⏱️ Time taken: {duration}s")
    print("🔹 Generated Code:\n", output[:700], "\n")

    results.append({
        "model": model_id,
        "prompt": prompt,
        "output": output,
        "time": duration
    })

clear_memory()

df_deepseek = pd.DataFrame(results)


keywords = {
    "prime": ["def", "for", "if", "return"],
    "Flask": ["Flask", "app.route", "return"],
    "SQL": ["SELECT", "FROM", "WHERE"],
    "HTML": ["<html>", "<body>", "<h1>"],
    "calculator": ["class", "def", "return"],
    "NumPy": ["import numpy", "np.mean", "np.std"],
    "Decision Tree": ["DecisionTreeClassifier", "fit", "predict"],
    "Java": ["public class", "void", "LinkedList"],
    "Fibonacci": ["def", "return", "recursion"],
    "matplotlib": ["import matplotlib", "plt.plot"]
}

scores = []
for i, row in df_deepseek.iterrows():
    topic = list(keywords.keys())[i % 10]
    score = any(kw.lower() in row["output"].lower() for kw in keywords[topic])
    scores.append(int(score))
df_deepseek["logic_correct"] = scores

accuracy = df_deepseek["logic_correct"].mean() * 100
print(f"\n🎯 DeepSeek-Coder Logic Accuracy: {accuracy:.1f}%")


plt.bar(["DeepSeek-Coder"], [accuracy], color='skyblue')
plt.title("Logic Accuracy - DeepSeek-Coder-1.3B")
plt.ylabel("Accuracy (%)")
plt.ylim(0, 100)
plt.show()

df_deepseek.to_csv("deepseek_results.csv", index=False)
print("\n💾 Saved: deepseek_results.csv")


# **Model 2: Phi-2 (microsoft/phi-2)**

In [ ]:

import time, torch, gc, pandas as pd, matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()


model_id = "microsoft/phi-2"
print(f"Loading {model_id} ...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, torch_dtype=torch.float16, device_map="auto")


results = []
for i, prompt in enumerate(prompts, 1):
    print(f"\n==============================")
    print(f" Prompt {i}: {prompt}")
    print(f"==============================")

    start = time.time()
    output = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.3)[0]['generated_text']
    end = time.time()

    duration = round(end - start, 2)
    print(f"Time taken: {duration}s")
    print(" Generated Code:\n", output[:700], "\n")

    results.append({
        "model": model_id,
        "prompt": prompt,
        "output": output,
        "time": duration
    })

clear_memory()


df_phi2 = pd.DataFrame(results)


keywords = {
    "prime": ["def", "for", "if", "return"],
    "Flask": ["Flask", "app.route", "return"],
    "SQL": ["SELECT", "FROM", "WHERE"],
    "HTML": ["<html>", "<body>", "<h1>"],
    "calculator": ["class", "def", "return"],
    "NumPy": ["import numpy", "np.mean", "np.std"],
    "Decision Tree": ["DecisionTreeClassifier", "fit", "predict"],
    "Java": ["public class", "void", "LinkedList"],
    "Fibonacci": ["def", "return", "recursion"],
    "matplotlib": ["import matplotlib", "plt.plot"]
}

scores = []
for i, row in df_phi2.iterrows():
    topic = list(keywords.keys())[i % 10]
    score = any(kw.lower() in row["output"].lower() for kw in keywords[topic])
    scores.append(int(score))
df_phi2["logic_correct"] = scores


accuracy = df_phi2["logic_correct"].mean() * 100
print(f"\n Phi-2 Logic Accuracy: {accuracy:.1f}%")


plt.bar(["Phi-2"], [accuracy], color='orange')
plt.title("Logic Accuracy - Phi-2")
plt.ylabel("Accuracy (%)")
plt.ylim(0, 100)
plt.show()


df_phi2.to_csv("phi2_results.csv", index=False)
print("\n Saved: phi2_results.csv")


# **Model 3 : Gemma-2B-IT**

In [ ]:
from huggingface_hub import login
login()


In [ ]:

import time, torch, gc, pandas as pd, matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()

model_id = "google/gemma-2b-it"
print(f"Loading {model_id} ...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, torch_dtype=torch.float16, device_map="auto")

results = []
for i, prompt in enumerate(prompts, 1):
    print(f"\n==============================")
    print(f"🧠 Prompt {i}: {prompt}")
    print(f"==============================")

    start = time.time()
    output = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.3)[0]['generated_text']
    end = time.time()

    duration = round(end - start, 2)
    print(f"⏱️ Time taken: {duration}s")
    print("🔹 Generated Code:\n", output[:700], "\n")

    results.append({
        "model": model_id,
        "prompt": prompt,
        "output": output,
        "time": duration
    })

clear_memory()


df_gemma = pd.DataFrame(results)


keywords = {
    "prime": ["def", "for", "if", "return"],
    "Flask": ["Flask", "app.route", "return"],
    "SQL": ["SELECT", "FROM", "WHERE"],
    "HTML": ["<html>", "<body>", "<h1>"],
    "calculator": ["class", "def", "return"],
    "NumPy": ["import numpy", "np.mean", "np.std"],
    "Decision Tree": ["DecisionTreeClassifier", "fit", "predict"],
    "Java": ["public class", "void", "LinkedList"],
    "Fibonacci": ["def", "return", "recursion"],
    "matplotlib": ["import matplotlib", "plt.plot"]
}

scores = []
for i, row in df_gemma.iterrows():
    topic = list(keywords.keys())[i % 10]
    score = any(kw.lower() in row["output"].lower() for kw in keywords[topic])
    scores.append(int(score))
df_gemma["logic_correct"] = scores


accuracy = df_gemma["logic_correct"].mean() * 100
print(f"\n Gemma-2B-IT Logic Accuracy: {accuracy:.1f}%")

plt.bar(["Gemma-2B-IT"], [accuracy], color='green')
plt.title("Logic Accuracy - Gemma-2B-IT")
plt.ylabel("Accuracy (%)")
plt.ylim(0, 100)
plt.show()


df_gemma.to_csv("gemma_results.csv", index=False)
print("\n Saved: gemma_results.csv")


##visualization Plot

In [ ]:


import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_deepseek = pd.read_csv("deepseek_results.csv")
df_phi2 = pd.read_csv("phi2_results.csv")
df_gemma = pd.read_csv("gemma_results.csv")


for df, name in zip(
    [df_deepseek, df_phi2, df_gemma],
    ["DeepSeek-Coder", "Phi-2", "Gemma-2B-IT"]
):
    if "model" not in df.columns:
        df["model"] = name


df_all = pd.concat([df_deepseek, df_phi2, df_gemma], ignore_index=True)


performance = df_all.groupby("model").agg({
    "logic_correct": "mean",
    "time": "mean"
}).reset_index()

performance["logic_correct"] = performance["logic_correct"] * 100
performance.rename(columns={"logic_correct": "Logic Accuracy (%)", "time": "Avg Time (s)"}, inplace=True)


print(" Model Performance Summary:\n")
print(performance)

plt.figure(figsize=(8,5))
plt.bar(performance["model"], performance["Logic Accuracy (%)"])
plt.title("Logic Accuracy Comparison Across Models")
plt.ylabel("Accuracy (%)")
plt.xticks(rotation=15)
plt.ylim(0, 100)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()
plt.figure(figsize=(8,5))
plt.bar(performance["model"], performance["Avg Time (s)"], color="orange")
plt.title("Average Generation Time per Prompt")
plt.ylabel("Time (seconds)")
plt.xticks(rotation=15)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()
plt.figure(figsize=(6,4))
sns.heatmap(performance.set_index("model"), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Model Performance Heatmap")
plt.show()


# **Model Ranking**

In [ ]:

acc_weight = 0.7
time_weight = 0.3


performance["Time Score"] = 100 * (1 - performance["Avg Time (s)"] / performance["Avg Time (s)"].max())
performance["Final Score"] = (
    performance["Logic Accuracy (%)"] * acc_weight +
    performance["Time Score"] * time_weight
)

performance = performance.sort_values("Final Score", ascending=False).reset_index(drop=True)
performance["Rank"] = performance.index + 1

print("\n Model Performance")
print(performance[["Rank", "model", "Logic Accuracy (%)", "Avg Time (s)", "Final Score"]])

plt.figure(figsize=(8,5))
plt.bar(performance["model"], performance["Final Score"], color="green")
plt.title("Final Composite Score (Accuracy + Speed)")
plt.ylabel("Score (0–100)")
plt.xticks(rotation=15)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()


# **UI 1 — Code Master**

In [ ]:

!pip install ipywidgets transformers accelerate --quiet

from IPython.display import display, Markdown, HTML
import ipywidgets as widgets
from transformers import pipeline
import torch

models = {
    "DeepSeek-Coder (1.3B)": "deepseek-ai/deepseek-coder-1.3b-instruct",
    "Phi-2 (2.7B)": "microsoft/phi-2",
    "Gemma-2B-IT": "google/gemma-2b-it"
}

# =========================
# UI Elements
# =========================
model_dropdown = widgets.Dropdown(
    options=list(models.keys()),
    description='Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

prompt_input = widgets.Textarea(
    value='Write a Python function to check if a number is prime.',
    placeholder='Enter your programming prompt here...',
    description='Prompt:',
    layout=widgets.Layout(width='100%', height='100px')
)

generate_button = widgets.Button(
    description=" Generate Code",
    button_style='success',
    layout=widgets.Layout(width='30%')
)

output_area = widgets.Output()


def generate_code(b):
    output_area.clear_output()
    with output_area:
        selected_model = models[model_dropdown.value]
        display(Markdown(f"### Model Selected: `{model_dropdown.value}`"))
        display(Markdown(f"### youe code Generating... Please wait ⏳"))
        try:
            generator = pipeline("text-generation", model=selected_model, torch_dtype=torch.bfloat16, device_map="auto")
            result = generator(prompt_input.value, max_new_tokens=200, do_sample=True, temperature=0.7)
            code = result[0]['generated_text']
            display(Markdown(f"###  Generated Code:"))
            display(HTML(f"<pre style='background:#f4f4f4;padding:10px;border-radius:10px'><code>{code}</code></pre>"))
        except Exception as e:
            display(Markdown(f" **Error:** {e}"))

generate_button.on_click(generate_code)


display(Markdown("(வணக்கம்)- Welcome to CodeMaster"))
display(model_dropdown, prompt_input, generate_button, output_area)
